In [ ]:
# !pip install langchain
# !pip install langchain-community
# !pip install pypdf

In [ ]:
import pprint

In [ ]:
# Import library
from langchain_community.document_loaders import PyPDFLoader,UnstructuredHTMLLoader

# Create a document loader for rag_paper.pdf
loader = PyPDFLoader('pdf_data.pdf')

# Load the document
pdf_data = loader.load()

print(pdf_data[0].page_content)
print(pdf_data[0].metadata)

In [ ]:
# !pip install unstructured

In [ ]:
# Import library
from langchain_community.document_loaders import UnstructuredHTMLLoader

# Load the document
html_data = UnstructuredHTMLLoader('html_data.html').load()

# Print the first document's content
print(html_data[0].page_content)

# Print the first document's metadata
print(html_data[0].metadata)

In [ ]:
from langchain_community.document_loaders import CSVLoader

loader = CSVLoader(file_path="csv_data.csv")
csv_data = loader.load()

print(csv_data[0].page_content)
print(csv_data[-1].metadata)

In [ ]:
# !pip install langchain_text_splitters

In [ ]:
from langchain_text_splitters import CharacterTextSplitter

text_splitter = CharacterTextSplitter(
    separator='.',
    chunk_size=75,  
    chunk_overlap=10  
)
# Split the text using text_splitter
chunks = text_splitter.split_text(csv_data[0].page_content)
pprint.pprint(chunks)
pprint.pprint([len(chunk) for chunk in chunks])

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    separators=['.'],
    chunk_size=75,
    chunk_overlap=10
)

# Split the text using text_splitter
chunks = text_splitter.split_text(csv_data[-1].page_content)
chunks = text_splitter.split_documents(pdf_data)
pprint.pprint(chunks)
pprint.pprint([len(chunk) for chunk in chunks])

In [ ]:
# !pip install langchain_chroma langchain_openai

In [ ]:
# !pip install langchain_community
# !pip install sentence-transformers

In [ ]:
# from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

# Create an instance of the OpenAIEmbeddings class

# embedding_model = OpenAIEmbeddings(
    # api_key="<OPENAI_API_TOKEN>",
    #   model='text-embedding-3-small')

from langchain_community.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Create a Chroma vector store and embed the chunks
vector_store = Chroma.from_documents(
    documents=chunks, 
    embedding=embedding_model
)

/tmp/ipykernel_13591/1908502660.py:12: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 103/103 [00:01<00:00, 52.64it/s]


In [ ]:
retriever = vector_store.as_retriever(
    search_kwargs={"k": 2},
    search_type="similarity",
    # search_type="mmr",
) 

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# prompt = ChatPromptTemplate.from_messages()
prompt = ChatPromptTemplate.from_template(
    "You are a helpful assistant. Answer the question based on the context provided.\n\nContext: {context}\n\nQuestion: {question}\n\nAnswer:"
)

In [ ]:
# llm = ChatOpenAI(model="gpt-3.5-turbo",api_key="your-api-key", temperature=0)

from transformers import pipeline
from langchain_community.llms import HuggingFacePipeline

pipe = pipeline(
    "text-generation",
    model="mistralai/Mistral-7B-Instruct-v0.2",
    device=0  # GPU (use -1 for CPU)
)

llm = HuggingFacePipeline(pipeline=pipe)

In [ ]:
from langchain_core.runnables import RunnablePassThrough
from langchain_core.output_parsers import StrOutputParser

In [ ]:
chain = (
    {"context": retriever, "question": RunnablePassThrough()}
    | prompt
    | llm
    | StrOutputParser()
)


In [ ]:
result = chain.invoke({"question": "What is the main topic of the document?"})
print(result)